# Heston Stochastic Volatility Model

This notebook implements the Heston model for European call option pricing including:

1. **Finite Difference Solver** 2D ADI (Alternating Direction Implicit) scheme with Crank-Nicolson splitting, solved on a log-price grid for numerical stability
2. **Analytic Formula** Characteristic function inversion via Fourier transform (Heston 1993).
3. **Convergence Study** We verify the convergence rate of the FD solver by comparing the grid resolutions to the resulting errors..



### Heston Model
Asset price $S$ and variance $V$ follow coupled SDEs:
$$dS = rS\,dt + \sqrt{V}S\,dW_1$$
$$dV = \kappa(\theta - V)\,dt + \sigma_v\sqrt{V}\,dW_2, \quad d W_1 d W_2 = \rho\,dt$$

Parameters: $\kappa$ (mean reversion speed), $\theta$ (long-run variance), $\sigma_v$ (vol of vol), $\rho$ (correlation), $v_0$ (initial variance).

By risk-neutral pricing, the value $U(S, V, t)$ of a European option satisfies 
the Heston PDE:

$$\frac{\partial U}{\partial t} + \frac{1}{2}VS^2\frac{\partial^2 U}{\partial S^2} 
+ \rho\sigma_v VS\frac{\partial^2 U}{\partial S \partial V} 
+ \frac{1}{2}\sigma_v^2 V\frac{\partial^2 U}{\partial V^2} 
+ rS\frac{\partial U}{\partial S} + \kappa(\theta - V)\frac{\partial U}{\partial V} - rU = 0$$

We assume the option price $U:[S_{min},S_{max}]\times[0,V_{max}]\times[0,T]\to \mathbb{R}$, with boundary conditions

$$U(S_{min},V,t)=0$$

$$U(S_{max},V,t)=S_{max}-Ke^{-r(T-t)}$$

$$U(S,0,t)=\max(S-Ke^{-r(T-t)},0)$$

$$U(S,V_{max},t)=S$$

$$U(S,V,T)=\max(S-K,0)$$

## 1. Setup

In [6]:
import numpy as np
import pandas as pd
from option_fd import thomas_solver
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import norm
from scipy.optimize import brentq, minimize
from scipy.interpolate import RegularGridInterpolator
import time
import yfinance as yf
from mpl_toolkits.mplot3d import Axes3D

## 2. Finite Difference Solver (ADI)

We introduce the change of variables $x = \log S$. Differentiating yields $$\frac{\partial x}{\partial S}=\frac{1}{S}$$

By the chain rule:
$$\frac{\partial U}{\partial S}=\frac{\partial U}{\partial x}\frac{\partial x}{\partial S}=\frac{1}{S}\frac{\partial U}{\partial x}$$

Similarly, 
$$\frac{\partial^2 U}{\partial S^2}=\frac{\partial }{\partial S}\left(\frac{\partial U}{\partial x}\frac{1}{S}\right)
=\frac{\partial }{\partial S}\left(\frac{\partial U}{\partial x}\right)\frac{1}{S}+\frac{\partial U}{\partial x}\left(-\frac{1}{S^2}\right)
=\frac{\partial }{\partial x}\left(\frac{\partial U}{\partial S}\right)\frac{1}{S}-\frac{1}{S^2}\frac{\partial U}{\partial x}=\frac{\partial }{\partial x}\left(\frac{\partial U}{\partial x}\frac{1}{S}\right)\frac{1}{S}-\frac{1}{S^2}\frac{\partial U}{\partial x}$$

In summary, we have 
$$\frac{\partial^2 U}{\partial S^2}=\frac{1}{S^2}\left(\frac{\partial^2 U}{\partial x^2}-\frac{\partial U}{\partial x}\right)$$

Substituting:
$$\frac{\partial U}{\partial t} + \frac{1}{2}VS^2\left[\frac{1}{S^2}\left(\frac{\partial^2 U}{\partial x^2}-\frac{\partial U}{\partial x}\right)\right] + \rho\sigma_v VS\frac{\partial }{\partial V}\left[\frac{1}{S}\frac{\partial U}{\partial x}\right] 
+ \frac{1}{2}\sigma_v^2 V\frac{\partial^2 U}{\partial V^2} 
+ rS\left[\frac{1}{S}\frac{\partial U}{\partial x}\right] + \kappa(\theta - V)\frac{\partial U}{\partial V} - rU = 0 $$

Therefore, Heston PDE in log-price coordinates:
$$\frac{\partial U}{\partial t} + \frac{1}{2}V\frac{\partial^2 U}{\partial x^2} + \left(r - \frac{1}{2}V\right)\frac{\partial U}{\partial x} + \frac{1}{2}\sigma_v^2 V\frac{\partial^2 U}{\partial V^2} + \kappa(\theta-V)\frac{\partial U}{\partial V} + \rho\sigma_v V\frac{\partial^2 U}{\partial x \partial V} - rU = 0$$
- **V-sweep**: implicit in $V$, correcting the explicit $V$ terms


**ADI splitting**: The 2D Crank-Nicolson system is split into two 1D tridiagonal solves per time step:
- **S-sweep**: implicit in $x$, explicit in $V$
- **V-sweep**: implicit in $V$, correcting the explicit $V$ terms

We assume the option price $U:[x_{min},x_{max}]\times[0,V_{max}]\times[0,T]\to \mathbb{R}$, where $x_{max}=\log(S_{max})$ and introduce another chosen cutoff $S_{min}>0$ so that $x_{min}=\log(S_{min})$ is defined. $V_{max}$ is our cutoff for the variance, and $T$ is the time length of the contract. 

The boundary conditions become:

$$U(x_{min},V,t)=0$$

$$U(x_{max},V,t)=e^{x_{max}}-Ke^{-r(T-t)}$$

$$U(x,0,t)=\max(e^x-Ke^{-r(T-t)},0)$$

$$U(x,V_{max},t)=e^x$$

$$U(x,V,T)=\max(e^x-K,0)$$








We have $\Delta x=\frac{x_{max}-x_{min}}{M}$, $\Delta V=\frac{V_{max}}{P}$, and $\Delta t=\frac{T}{N}$

We write

$x_i=x_{min}+i\Delta x$ for $i=0,...,M$, 

$V_j=j\Delta V$ for $j=0,...,P$,

$t_n=n\Delta t$ for $n=0,...,N$

We split our PDE 

$$\frac{\partial U}{\partial t} + \frac{1}{2}V\frac{\partial^2 U}{\partial x^2} + \left(r - \frac{1}{2}V\right)\frac{\partial U}{\partial x} + \frac{1}{2}\sigma_v^2 V\frac{\partial^2 U}{\partial V^2} + \kappa(\theta-V)\frac{\partial U}{\partial V} + \rho\sigma_v V\frac{\partial^2 U}{\partial x \partial V} - rU = 0$$

into spatial operators:

$$L_x U+L_V U+L_{\text{mix}} U=-\frac{\partial U}{\partial t}$$

where

$$L_x U=  \frac{1}{2}V\frac{\partial^2 U}{\partial x^2} + \left(r - \frac{1}{2}V\right)\frac{\partial U}{\partial x}- rU$$

$$L_V U= \frac{1}{2}\sigma_v^2 V\frac{\partial^2 U}{\partial V^2} + \kappa(\theta-V)\frac{\partial U}{\partial V}$$

$$L_{\text{mix}} U = \rho\sigma_v V\frac{\partial^2 U}{\partial x \partial V}$$

We utilize the difference quotient formulas:
$$\frac{\partial U}{\partial x}(x_i,V_j,t_n)=\frac{U(x_{i+1},V_j,t_n)-U(x_{i-1},V_j, t_n)}{2\Delta x}+ \mathcal{O}((\Delta x)^2)$$

$$\frac{\partial^2 U}{\partial x^2}(x_i,V_j,t_n)=\frac{U(x_{i+1},V_j,t_n)-2U(x_i,V_j,t_n)+U(x_{i-1},V_j,t_n)}{(\Delta x)^2}+ \mathcal{O}((\Delta x)^2)$$

$$\frac{\partial U}{\partial V}(x_i,V_j,t_n)=\frac{U(x_i,V_{j+1},t_n)-U(x_i,V_{j-1},t_n)}{2\Delta V}+ \mathcal{O}((\Delta V)^2)$$

$$\frac{\partial^2 U}{\partial V^2}(x_i,V_j,t_n)=\frac{U(x_i,V_{j+1},t_n)-2U(x_i,V_j,t_n)+U(x_i,V_{j-1},t_n)}{(\Delta V)^2}+ \mathcal{O}((\Delta V)^2)$$

$$\frac{\partial U}{\partial t}(x_i,V_j,t_n)=\frac{U(x_i,V_j,t_{n+1})-U(x_i,V_j,t_n)}{\Delta t}+ \mathcal{O}(\Delta t)$$

and finally the mixed partial:

$$\frac{\partial^2 U}{\partial V \partial x}(x_i,V_j,t_n)=\frac{\partial }{\partial V}\left[\frac{\partial U}{\partial x}(x_i,V_j,t_n)\right]\approx \frac{\partial }{\partial V}\left[ \frac{U(x_{i+1},V_j,t_n)-U(x_{i-1},V_j, t_n)}{2\Delta x}\right]$$

$$\approx\frac{1}{2\Delta x}\frac{\partial }{\partial V}\left[U(x_{i+1},V_j,t_n)-U(x_{i-1},V_j, t_n)\right]$$

$$\approx\frac{1}{2\Delta x}\frac{\left[U(x_{i+1},V_{j+1},t_n)-U(x_{i+1},V_{j-1},t_n)\right]-\left[ U(x_{i-1},V_{j+1}, t_n)-U(x_{i-1},V_{j-1}, t_n)\right]}{2\Delta V}$$

So that 

$$\frac{\partial^2 U}{\partial V \partial x}(x_i,V_j,t_n)=\frac{U(x_{i+1},V_{j+1},t_n)-U(x_{i+1},V_{j-1},t_n)-U(x_{i-1},V_{j+1}, t_n)+U(x_{i-1},V_{j-1}, t_n)}{4\Delta x\Delta V}+\mathcal{O}((\Delta x)^2+(\Delta V)^2)$$










Next we discretize the PDE. We have,

$$L_x U(x_i,V_j,t_n)=  \frac{1}{2}V_j\frac{\partial^2 U}{\partial x^2}(x_i,V_j,t_n) + \left(r - \frac{1}{2}V_j\right)\frac{\partial U}{\partial x}(x_i,V_j,t_n)- rU(x_i,V_j,t_n)$$

$$\approx \frac{1}{2}V_j\left(\frac{U(x_{i+1},V_j,t_n)-2U(x_i,V_j,t_n)+U(x_{i-1},V_j,t_n)}{(\Delta x)^2}\right)+ \left(r - \frac{1}{2}V_j\right)\left(\frac{U(x_{i+1},V_j,t_n)-U(x_{i-1},V_j, t_n)}{2\Delta x}\right)- rU(x_i,V_j,t_n)
$$

$$=\left(\alpha_j-\beta_j\right)U(x_{i-1},V_j,t_n)+\left(-2\alpha_j-r\right)U(x_{i},V_j,t_n)+\left(\alpha_j+\beta_j\right)U(x_{i+1},V_j,t_n)$$

where

$$\alpha_j=\frac{V_j}{2(\Delta x)^2}$$

and 

$$\beta_j=\frac{r-\frac{1}{2}V_j}{2\Delta x}$$.


Similarly, 

$$L_V U(x_i,V_j,t_n)=\frac{1}{2}\sigma_v^2 V_j\frac{\partial^2 U}{\partial V^2}(x_i,V_j,t_n) + \kappa(\theta-V_j)\frac{\partial U}{\partial V}(x_i,V_j,t_n)$$

$$\approx\frac{1}{2}\sigma_v^2 V_j\left(\frac{U(x_i,V_{j+1},t_n)-2U(x_i,V_j,t_n)+U(x_i,V_{j-1},t_n)}{(\Delta V)^2}\right)+\kappa(\theta-V_j)\left(\frac{U(x_i,V_{j+1},t_n)-U(x_i,V_{j-1},t_n)}{2\Delta V}\right)$$

$$=\left(\gamma_j-\delta_j\right)U(x_i,V_{j-1},t_n)+\left(-2\gamma_j\right)U(x_i,V_{j},t_n)+\left(\gamma_j+\delta_j\right)U(x_i,V_{j+1},t_n)$$

where

$$\gamma_j=\frac{\sigma_v^2V_j}{2(\Delta V)^2}$$

and

$$\delta_j=\frac{\kappa(\theta-V_j)}{2\Delta V}$$

We treat $L_x$ and $L_V$ implicitly using Crank-Nicolson averaging, and $L_{\text{mix}}$ and $\frac{\partial}{\partial t}$ explicitly, as an implicit treatment of the mixed term would destroy the tridiagonal structure of each sweep. Set

$$-\frac{\partial U}{\partial t}(x_i,V_j,t_n)=\frac{1}{2}\left[L_x U(x_i,V_j,t_n)+L_x U(x_i,V_j,t_{n+1})\right]+\frac{1}{2}\left[L_V U(x_i,V_j,t_n)+L_V U(x_i,V_j,t_{n+1})\right]+L_{\text{mix}}U(x_i,V_j,t_{n+1})$$


Discretizing $\frac{\partial U}{\partial t}$ and multiplying by $\Delta t$, we obtain the compactly stated: 


$$\left(I - \frac{\Delta t}{2}L_x - \frac{\Delta t}{2}L_V\right)U(x_i,V_j,t_n) = \left(I + \frac{\Delta t}{2}L_x + \frac{\Delta t}{2}L_V\right)U(x_i,V_j,t_{n+1}) + (\Delta t) L_{mix}U(x_i,V_j,t_{n+1})$$

Since solving this 2D system directly is expensive, we instead split into two successive tridiagonal systems by introducing an intermediate value $U^*$:
#### S-sweep
$$\left(I - \frac{\Delta t}{2}L_x\right)U^*(x_i,V_j) = \left(I + \frac{\Delta t}{2}L_x+\Delta t\, L_V\right)U(x_i,V_j,t_{n+1}) + \Delta t\, L_{\text{mix}}U(x_i,V_j,t_{n+1})$$

and 

#### V-sweep
$$\left(I - \frac{\Delta t}{2}L_V\right)U(x_i,V_j,t_n) = U^*(x_i,V_j) - \frac{\Delta t}{2}L_V U(x_i,V_j,t_{n+1})$$

Observe that substituting $U^*$ from the V-sweep into the S-sweep and dropping the $\mathcal{O}((\Delta t)^2)$ cross terms $\frac{(\Delta t)^2}{4}L_xL_V$ term recovers the original 2D system, and so the splitting is consistent with the second order accuracy in Crank-Nicolson.

# S-sweep
$$\left(I - \frac{\Delta t}{2}L_x\right)U^*(x_i,V_j) = \left(I + \frac{\Delta t}{2}L_x+\Delta t\, L_V\right)U(x_i,V_j,t_{n+1}) + \Delta t\, L_{\text{mix}}U(x_i,V_j,t_{n+1})$$
Discretizing:
$$\begin{aligned}
&U^*(x_i,V_j) - \frac{\Delta t}{2}\left[\left(\alpha_j-\beta_j\right)U^*(x_{i-1},V_j)+\left(-2\alpha_j-r\right)U^*(x_{i},V_j)+\left(\alpha_j+\beta_j\right)U^*(x_{i+1},V_j)\right] =\\
&U(x_i,V_j,t_{n+1}) + \frac{\Delta t}{2}\left[\left(\alpha_j-\beta_j\right)U(x_{i-1},V_j,t_{n+1})+\left(-2\alpha_j-r\right)U(x_{i},V_j,t_{n+1})+\left(\alpha_j+\beta_j\right)U(x_{i+1},V_j,t_{n+1})\right] \\
&+ \Delta t\left[\left(\gamma_j-\delta_j\right)U(x_i,V_{j-1},t_{n+1})+\left(-2\gamma_j\right)U(x_i,V_{j},t_{n+1})+\left(\gamma_j+\delta_j\right)U(x_i,V_{j+1},t_{n+1})\right]+\Delta t\, L_{\text{mix}}U(x_i,V_j,t_{n+1})
\end{aligned}$$
Collecting the coefficients, it becomes:
$$\begin{aligned}
&A^-_j U^*(x_{i-1},V_j)+A^0_jU^*(x_{i},V_j)+A^+_j U^*(x_{i+1},V_j) \\
&=B^-_j U(x_{i-1},V_j,t_{n+1})+B^0_jU(x_{i},V_j,t_{n+1})+B^+_j U(x_{i+1},V_j,t_{n+1}) \\
&+ C_j^-U(x_i,V_{j-1},t_{n+1})+C_j^0U(x_i,V_{j},t_{n+1})+C_j^+U(x_i,V_{j+1},t_{n+1})+\Delta t\, L_{\text{mix}}U(x_i,V_j,t_{n+1})
\end{aligned}$$
where 
$$A_j^- = -\frac{\Delta t}{2}(\alpha_j - \beta_j), \qquad A_j^0 = 1 + \Delta t\,\alpha_j + \frac{\Delta t}{2}r, \qquad A_j^+ = -\frac{\Delta t}{2}(\alpha_j + \beta_j)$$
$$B_j^- = \frac{\Delta t}{2}(\alpha_j - \beta_j), \qquad B_j^0 = 1 - \Delta t\,\alpha_j - \frac{\Delta t}{2}r, \qquad B_j^+ = \frac{\Delta t}{2}(\alpha_j + \beta_j)$$
$$C_j^- = \Delta t(\gamma_j - \delta_j), \qquad C_j^0 = -2\Delta t\,\gamma_j, \qquad C_j^+ = \Delta t(\gamma_j + \delta_j)$$

Note the term $$L_{\text{mix}}U(x_i,V_j,t_{n+1})=\frac{\rho\sigma_v V_j}{4\Delta x\Delta V}\left[U(x_{i+1},V_{j+1},t_{n+1})-U(x_{i+1},V_{j-1},t_{n+1})-U(x_{i-1},V_{j+1}, t_{n+1})+U(x_{i-1},V_{j-1}, t_{n+1})\right]$$
# V-sweep
$$\left(I - \frac{\Delta t}{2}L_V\right)U(x_i,V_j,t_n) = U^*(x_i,V_j) - \frac{\Delta t}{2}L_V U(x_i,V_j,t_{n+1})$$
Discretizing:
$$\begin{aligned}
&U(x_i,V_j,t_n) - \frac{\Delta t}{2}\left[(\gamma_j-\delta_j)\,U(x_i,V_{j-1},t_n)+(-2\gamma_j)\,U(x_i,V_j,t_n)+(\gamma_j+\delta_j)\,U(x_i,V_{j+1},t_n)\right] \\
&= U^*(x_i,V_j) - \frac{\Delta t}{2}\left[(\gamma_j-\delta_j)\,U(x_i,V_{j-1},t_{n+1})+(-2\gamma_j)\,U(x_i,V_j,t_{n+1})+(\gamma_j+\delta_j)\,U(x_i,V_{j+1},t_{n+1})\right]
\end{aligned}$$
Collecting the coefficients:
$$D^-_j\,U(x_i,V_{j-1},t_n) + D^0_j\,U(x_i,V_j,t_n) + D^+_j\,U(x_i,V_{j+1},t_n) = U^*(x_i,V_j) + E^-_j\,U(x_i,V_{j-1},t_{n+1}) + E^0_j\,U(x_i,V_j,t_{n+1}) + E^+_j\,U(x_i,V_{j+1},t_{n+1})$$
where
$$D_j^- = -\frac{\Delta t}{2}(\gamma_j - \delta_j), \qquad D_j^0 = 1 + \Delta t\,\gamma_j, \qquad D_j^+ = -\frac{\Delta t}{2}(\gamma_j + \delta_j)$$
$$E_j^- = -\frac{\Delta t}{2}(\gamma_j - \delta_j), \qquad E_j^0 = \Delta t\,\gamma_j, \qquad E_j^+ = -\frac{\Delta t}{2}(\gamma_j + \delta_j)$$

For each time step $n$, the S-sweep solves a tridiagonal system in $i$ for each fixed $j$, and the V-sweep solves a tridiagonal system in $j$ for each fixed $i$, with known boundary values folded into the right-hand side.


In [9]:
def heston_fd(M, N, P, S_0, K, r, T, kappa, theta, sigma_v, rho, v0,
              x_min=np.log(10.0), x_max=np.log(600.0), V_max=1.0):
    dx = (x_max - x_min) / M
    dt = T / N
    dV = V_max / P

    x = np.linspace(x_min, x_max, M+1)
    V = np.linspace(0, V_max, P+1)
    t = np.linspace(0, T, N+1)
    S = np.exp(x)

    XX, VV = np.meshgrid(x[1:M], V[1:P], indexing='ij')

    # Terminal condition (call payoff)
    U = np.zeros((M+1, P+1))
    for j in range(P+1):
        U[:, j] = np.maximum(S - K, 0.0)

    # Spatial coefficients (shape: (M-1)x(P-1))
    alpha = 0.5 * VV / dx**2
    beta  = (r - 0.5*VV) / (2*dx)
    gamma = 0.5 * sigma_v**2 * VV / dV**2
    delta = kappa * (theta - VV) / (2*dV)
    mixed = rho * sigma_v * VV / (4*dx*dV)

    # S-sweep LHS
    A_minus = -0.5*dt*(alpha - beta)
    A_zero  =  1 + dt*alpha + 0.5*dt*r
    A_plus  = -0.5*dt*(alpha + beta)

    # S-sweep RHS
    B_minus =  0.5*dt*(alpha - beta)
    B_zero  =  1 - dt*alpha - 0.5*dt*r
    B_plus  =  0.5*dt*(alpha + beta)

    # S-sweep RHS (explicit)
    C_minus =  dt*(gamma - delta)
    C_zero  = -2*dt*gamma
    C_plus  =  dt*(gamma + delta)

    # V-sweep LHS
    D_minus = -0.5*dt*(gamma - delta)
    D_zero  =  1 + dt*gamma
    D_plus  = -0.5*dt*(gamma + delta)

    # V-sweep RHS (explicit)
    E_minus = -0.5*dt*(gamma - delta)
    E_zero  =  dt*gamma
    E_plus  = -0.5*dt*(gamma + delta)

    t0 = time.time()

    for n_idx in range(N-1, -1, -1):
        U_old = U.copy()
        t_n = t[n_idx]

        # Boundary conditions at time t_n
        U[0,  :] = 0.0
        U[M,  :] = np.exp(x_max) - K*np.exp(-r*t_n)
        U[1:M, 0] = np.maximum(S[1:M] - K*np.exp(-r*t_n), 0.0)
        U[1:M, P] = S[1:M]

        # Mixed derivative 
        mixed_rhs = np.zeros((M+1, P+1))
        mixed_rhs[1:M, 1:P] = dt * mixed * (
              U_old[2:M+1, 2:P+1]
            - U_old[2:M+1, 0:P-1]
            - U_old[0:M-1, 2:P+1]
            + U_old[0:M-1, 0:P-1]
        )

        # S-sweep
        U_star = np.zeros((M+1, P+1))
        U_star[0, :] = U[0, :];  U_star[M, :] = U[M, :]
        U_star[:, 0] = U[:, 0];  U_star[:, P] = U[:, P]

        for j in range(1, P):
            jj = j - 1
            rhs = (B_minus[:, jj] * U_old[0:M-1, j] +
                   B_zero[:,  jj] * U_old[1:M,   j] +
                   B_plus[:,  jj] * U_old[2:M+1, j])
            rhs += (C_minus[:, jj] * U_old[1:M, j-1] +
                    C_zero[:,  jj] * U_old[1:M, j]   +
                    C_plus[:,  jj] * U_old[1:M, j+1])
            rhs += mixed_rhs[1:M, j]
            rhs[0]  -= A_minus[0,  jj] * U_star[0, j]
            rhs[-1] -= A_plus[-1,  jj] * U_star[M, j]
            U_star[1:M, j] = thomas_solver(A_minus[1:, jj], A_zero[:, jj], A_plus[:-1, jj], rhs)

        # V-sweep
        for i in range(1, M):
            ii = i - 1
            rhs = U_star[i, 1:P].copy()
            rhs += (E_minus[ii, :] * U_old[i, 0:P-1] +
                    E_zero[ii,  :] * U_old[i, 1:P]   +
                    E_plus[ii,  :] * U_old[i, 2:P+1])
            rhs[0]  -= D_minus[ii, 0]  * U[i, 0]
            rhs[-1] -= D_plus[ii,  -1] * U[i, P]
            U[i, 1:P] = thomas_solver(D_minus[ii, 1:], D_zero[ii, :], D_plus[ii, :-1], rhs)

    elapsed = time.time() - t0
    interp = RegularGridInterpolator((S, V), U)
    return interp([[S_0, v0]])[0], elapsed


In [10]:
# Model parameters
S_0     = 100
K       = 100
r       = 0.05
T       = 1.0

# Heston parameters
kappa   = 2.0
theta   = 0.04
sigma_v = 0.3
rho     = -0.7
v0      = 0.04

fd_price, elapsed = heston_fd(200, 2000, 100, S_0, K, r, T, kappa, theta, sigma_v, rho, v0)
print(f"FD price: {fd_price:.4f}  ({elapsed:.1f}s)")


FD price: 10.4587  (93.2s)


## 3. Analytic Formula (Heston 1993)

The call price is $C = S_0 P_1 - K e^{-rT} P_2$ where $P_1$, $P_2$ are risk-neutral probabilities
obtained by Fourier inversion of the characteristic function.

In [4]:
def heston_call_analytic(S0, K, T, r, kappa, theta, sigma_v, rho, v0):
    def char_func(phi, j):
        u = 0.5 if j == 1 else -0.5
        b = (kappa - rho*sigma_v) if j == 1 else kappa
        x = np.log(S0)
        d = np.sqrt((rho*sigma_v*phi*1j - b)**2 - sigma_v**2*(2*u*phi*1j - phi**2))
        g = (b - rho*sigma_v*phi*1j + d) / (b - rho*sigma_v*phi*1j - d)
        C = (r*phi*1j*T
             + (kappa*theta/sigma_v**2)
             * ((b - rho*sigma_v*phi*1j + d)*T
                - 2*np.log((1 - g*np.exp(d*T))/(1 - g))))
        D = ((b - rho*sigma_v*phi*1j + d)/sigma_v**2
             * (1 - np.exp(d*T))/(1 - g*np.exp(d*T)))
        return np.exp(C + D*v0 + 1j*phi*x)

    def integrand(phi, j):
        return np.real(np.exp(-1j*phi*np.log(K)) * char_func(phi, j) / (1j*phi))

    P1 = 0.5 + (1/np.pi) * quad(lambda phi: integrand(phi, 1), 1e-9, 200)[0]
    P2 = 0.5 + (1/np.pi) * quad(lambda phi: integrand(phi, 2), 1e-9, 200)[0]
    return S0*P1 - K*np.exp(-r*T)*P2


analytic_price = heston_call_analytic(S_0, K, T, r, kappa, theta, sigma_v, rho, v0)
print(f"Analytic price: {analytic_price:.4f}")
print(f"FD price:       {fd_price:.4f}")
print(f"Difference:     {abs(fd_price - analytic_price):.4f}")

Analytic price: 10.3942
FD price:       10.4587
Difference:     0.0645


## 3a. Grid Convergence Study

We verify the FD solver by doubling the grid resolution and checking that the error shrinks at the expected second-order rate.


In [5]:
grids = [
    (50,   250,  25),
    (100,  500,  50),
    (200,  2000, 100),
]

analytic_price = heston_call_analytic(S_0, K, T, r, kappa, theta, sigma_v, rho, v0)
print(f"Analytic price: {analytic_price:.6f}\n")
print(f"{'M':>5s} {'N':>6s} {'P':>5s} {'FD Price':>10s} {'Error':>10s} {'Rel Err':>10s} {'Rate':>6s} {'Time (s)':>8s}")
print("-" * 65)

prev_err = None
for (M_c, N_c, P_c) in grids:
    fd_p, el = heston_fd(M_c, N_c, P_c, S_0, K, r, T, kappa, theta, sigma_v, rho, v0)
    err = abs(fd_p - analytic_price)
    rel = err / analytic_price * 100
    if prev_err is not None and err > 0:
        rate = np.log2(prev_err / err)
        print(f"{M_c:5d} {N_c:6d} {P_c:5d} {fd_p:10.4f} {err:10.4f} {rel:9.3f}% {rate:6.2f} {el:8.1f}")
    else:
        print(f"{M_c:5d} {N_c:6d} {P_c:5d} {fd_p:10.4f} {err:10.4f} {rel:9.3f}%    --- {el:8.1f}")
    prev_err = err


Analytic price: 10.394219

    M      N     P   FD Price      Error    Rel Err   Rate Time (s)
-----------------------------------------------------------------
   50    250    25     8.7009     1.6933    16.291%    ---      0.9
  100    500    50    10.0801     0.3142     3.022%   2.43      6.2
  200   2000   100    10.4587     0.0645     0.620%   2.29     95.1
